# Silver Transformation - dimcostcenter

This notebook builds the `dimcostcenter` dataset in the Silver layer.



## Run Shared Libraries



## Processing Step



In [ ]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimcostcenter"

## Read Source Tables



In [ ]:
costcenterDf = spark.table("bronze.costcenter")

In [ ]:
costcenterDf.printSchema()

## Build Silver Dataset



In [ ]:
dimcostcenterDf = costcenterDf.filter(costcenterDf.RecordId.isNotNull()
    ).select(
        costcenterDf.CostCenterNumber,
        F.when(costcenterDf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(costcenterDf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
        F.from_utc_timestamp(costcenterDf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
        costcenterDf.Vat,
        costcenterDf.RecordId.alias("CostCenterRecordId")
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("CostCenterHashKey", F.xxhash64("CostCenterRecordId")
    )

display(dimcostcenterDf)


## Prepare Final DataFrame



In [ ]:
df_final = dimcostcenterDf


## Verify Source Schema



In [ ]:
saveDeltaTableToCatalog(df_final,"silver",Entity)
